In [17]:
import numpy as np
from tqdm.notebook import tqdm

In [3]:
pred = np.load('../insightface/recognition/_evaluation_/ijb/IJBC_result/vitl_depth36.npy')
label = np.load('../insightface/recognition/_evaluation_/ijb/IJBC_result/label.npy')

pred.shape, label.shape

((15658489,), (15658489,))

In [4]:
pred.min(), pred.max()

(-0.2743083530861531, 0.996352181050754)

In [5]:
(pred > 0).sum()

10153036

In [7]:
import pickle

with open('../insightface/recognition/_evaluation_/ijb/IJBC_result/p1_p2.pkl', 'rb') as f:
    p1_p2 = pickle.load(f)
p1 = p1_p2['p1']
p2 = p1_p2['p2']

p1[:5], p1[-5:]

(array([1, 1, 1, 1, 1]), array([171707, 171707, 171707, 171707, 171707]))

In [19]:
p1_p2= np.stack([p1, p2], axis = 1)
p1_p2.shape

(15658489, 2)

In [11]:
unique_p = np.unique(p1_p2)
unique_p

array([     1,      2,      3, ..., 187953, 187954, 187955])

In [24]:
d = {}
for i, (p1, p2) in tqdm(enumerate(p1_p2), total = len(p1_p2)):
    p1 = int(p1)
    p2 = int(p2)
    if p1 not in d:
        d[p1] = [[], []]
    if p2 not in d:
        d[p2] = [[], []]

    _score = float(pred[i])
    _label = int(label[i])

    d[p1][_label].append(_score)
    d[p2][_label].append(_score)
    
len(d)

  0%|          | 0/15658489 [00:00<?, ?it/s]

23124

In [26]:
d[1]

[[0.00430923371053378,
  -0.03169234486194536,
  0.006020785845024197,
  -0.008261700367733802,
  -0.02040741322657682,
  -0.2003993993697764,
  -0.033518605007281044,
  0.05365955750885501,
  0.08565155802310054,
  0.11939131089288263,
  -0.0047695468260959845,
  0.10156461034838443,
  -0.01252665934073805,
  -0.03058722381160972,
  0.03777024874896972,
  0.010329943326173706,
  -0.02466717299570747,
  0.05815781873095928,
  0.06387514996145542,
  0.12162277252188136,
  -0.07728575569998891,
  0.0402605337343552,
  -0.013426733092917906,
  0.07255530021414915,
  -0.025464454913661005,
  0.07553640886727454,
  0.020003995313439053,
  -0.06969170742934205,
  0.008894155952732657,
  0.04328624914460822,
  -0.032552584684645454,
  -0.020667885063598828,
  0.04555426576968882,
  0.040497879933265424,
  0.063464329362694,
  0.045572873891966764,
  0.005880878154936722,
  0.05021404777996279,
  0.047310191766680165,
  -0.045533457504809305,
  -0.022552762605818073,
  0.001701977303778926,
  

In [29]:
pred2 = []
label2 = []

for p, meta in d.items():
    match_scores = meta[1]
    unmatch_scores = meta[0]
    # print(match_scores)
    # print(unmatch_scores)
    if match_scores:
        pred2.append(min(match_scores))
        label2.append(1)
    if unmatch_scores:
        pred2.append(max(unmatch_scores))
        label2.append(0)

len(pred2), len(label2)

(46212, 46212)

In [30]:
pred2[:10], label2[:10]

([0.7388475002960911,
  0.1855327552422184,
  0.7388475002960911,
  0.1867109449556555,
  0.7659897291660798,
  0.22144305260021419,
  0.769981238229102,
  0.20700896442564054,
  0.7593151323935643,
  0.22282882543941934],
 [1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

In [31]:
import numpy as np

def compute_far_frr(pred, gt, targets=(1e-2, 1e-3, 1e-4, 1e-5)):
    pred = np.asarray(pred).astype(float)
    gt   = np.asarray(gt).astype(int)
    
    # Sort thresholds from high→low
    order = np.argsort(-pred)
    pred_sorted = pred[order]
    gt_sorted   = gt[order]
    
    # Cumulative TP/FP as threshold decreases
    cum_pos = (gt_sorted == 1).cumsum()
    cum_neg = (gt_sorted == 0).cumsum()
    
    total_pos = (gt == 1).sum()
    total_neg = (gt == 0).sum()
    
    # At each threshold:
    #   TP = cum_pos
    #   FP = cum_neg
    #   FRR = FN / P = (P - TP) / P
    #   FAR = FP / N
    TP  = cum_pos
    FP  = cum_neg
    FRR = (total_pos - TP) / total_pos
    FAR = FP / total_neg

    thresholds = pred_sorted

    def find_metric(target, axis):
        """
        axis = 'FRR' or 'FAR'
        Returns: (value_at_target, threshold)
        """
        arr = FRR if axis == "FRR" else FAR

        # Find threshold where metric crosses target
        idx = np.argmin(np.abs(arr - target))

        return arr[idx], thresholds[idx]

    result = {
        "FAR@FRR": {},
        "FRR@FAR": {},
    }

    for t in targets:
        frr_val, thr_frr = find_metric(t, "FRR")   # FAR@FRR target
        far_val, thr_far = find_metric(t, "FAR")   # FRR@FAR target

        result["FAR@FRR"][t] = {"FAR": float(FAR[np.argmin(np.abs(FRR - t))]),
                                "threshold": float(thr_frr)}

        result["FRR@FAR"][t] = {"FRR": float(FRR[np.argmin(np.abs(FAR - t))]),
                                "threshold": float(thr_far)}

    return result


def pretty_print_far_frr(result):
    print("\n" + "="*70)
    print("                 FAR @ Target FRR".center(70))
    print("="*70)
    print(f"{'Target FRR':>12} | {'Actual FAR':>12} | {'Threshold':>12}")
    print("-"*70)

    for tgt, vals in result["FAR@FRR"].items():
        print(f"{tgt:12.1e} | {vals['FAR']:12.6f} | {vals['threshold']:12.6f}")

    print("\n" + "="*70)
    print("                 FRR @ Target FAR".center(70))
    print("="*70)
    print(f"{'Target FAR':>12} | {'Actual FRR':>12} | {'Threshold':>12}")
    print("-"*70)

    for tgt, vals in result["FRR@FAR"].items():
        print(f"{tgt:12.1e} | {vals['FRR']:12.6f} | {vals['threshold']:12.6f}")

    print("="*70 + "\n")

In [32]:
result = compute_far_frr(pred2, label2, targets=(1e-1, 5e-2, 4e-2, 3e-2, 2e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5, 5e-6, 1e-6))
print(result)
pretty_print_far_frr(result)

{'FAR@FRR': {0.1: {'FAR': 0.001427088738972496, 'threshold': 0.5448542446249465}, 0.05: {'FAR': 0.003719079743988929, 'threshold': 0.43193992012371807}, 0.04: {'FAR': 0.007740875281093237, 'threshold': 0.38890552697903735}, 0.03: {'FAR': 0.02815256875973015, 'threshold': 0.32625860281530716}, 0.02: {'FAR': 0.6222539353053105, 'threshold': 0.2068158085601891}, 0.01: {'FAR': 1.0, 'threshold': 0.09524216378520665}, 0.005: {'FAR': 1.0, 'threshold': 0.032915348163915364}, 0.001: {'FAR': 1.0, 'threshold': -0.07477669501134122}, 0.0005: {'FAR': 1.0, 'threshold': -0.08923573852362421}, 0.0001: {'FAR': 1.0, 'threshold': -0.16034376088253924}, 5e-05: {'FAR': 1.0, 'threshold': -0.17882033415533383}, 1e-05: {'FAR': 1.0, 'threshold': -0.17882033415533383}, 5e-06: {'FAR': 1.0, 'threshold': -0.17882033415533383}, 1e-06: {'FAR': 1.0, 'threshold': -0.17882033415533383}}, 'FRR@FAR': {0.1: {'FRR': 0.02585758835758836, 'threshold': 0.27750745802067844}, 0.05: {'FRR': 0.028196465696465698, 'threshold': 0.3

In [33]:
result = compute_far_frr(pred, label, targets=(1e-1, 5e-2, 4e-2, 3e-2, 2e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5, 5e-6, 1e-6))
print(result)
pretty_print_far_frr(result)

{'FAR@FRR': {0.1: {'FAR': 8.312588097448086e-07, 'threshold': 0.5791563051165646}, 0.05: {'FAR': 2.4298334438694407e-06, 'threshold': 0.478619436858489}, 0.04: {'FAR': 3.5168641950741903e-06, 'threshold': 0.44433332064748365}, 0.03: {'FAR': 7.737101229163219e-06, 'threshold': 0.39040450184445796}, 0.02: {'FAR': 7.922535886721676e-05, 'threshold': 0.2915366554486919}, 0.01: {'FAR': 0.018137299912807346, 'threshold': 0.14413640725307936}, 0.005: {'FAR': 0.2058208322665512, 'threshold': 0.06794208092302698}, 0.001: {'FAR': 0.940312867911952, 'threshold': -0.06549653791015511}, 0.0005: {'FAR': 0.975506575512957, 'threshold': -0.08832802414290623}, 0.0001: {'FAR': 0.9980250569540171, 'threshold': -0.13908076205282072}, 5e-05: {'FAR': 0.9994465095186806, 'threshold': -0.16034376088253924}, 1e-05: {'FAR': 0.9998288246281779, 'threshold': -0.17882033415533383}, 5e-06: {'FAR': 0.9998288246281779, 'threshold': -0.17882033415533383}, 1e-06: {'FAR': 0.9998288246281779, 'threshold': -0.178820334155